# 🤖 Buổi 6/9 — Model Training: Linear → Tree → Ensemble
### Từng bước xây dựng và cải thiện model dự đoán giá nhà!

---

> **📍 Bạn đang ở đây trong lộ trình 9 buổi:**
> ```
> Buổi 1 ✅ Giới thiệu dự án & lộ trình
> Buổi 2 ✅ Setup môi trường & khám phá SalePrice
> Buổi 3 ✅ EDA — Phân tích missing, tương quan, outliers
> Buổi 4 ✅ Data Cleaning & Preprocessing
> Buổi 5 ✅ Feature Engineering
> Buổi 6 🔄 Model Training (Linear + Tree + Stacking)  ← BẠN ĐANG Ở ĐÂY
> Buổi 7 ⏳ Model Evaluation (Đánh giá toàn diện)
> Buổi 8 ⏳ Submit Kaggle (tạo file & nộp bài)
> Buổi 9 ⏳ Quiz Tốt Nghiệp (20 câu ôn tập)
> ```

---

### 📋 Nội Dung Buổi 6

| # | Task | Nội dung |
|---|------|----------|
| 1 | 📐 Linear Models | Ridge + Lasso làm baseline nhanh |
| 2 | 🌳 Tree-Based Models | Random Forest + XGBoost |
| 3 | 🎛️ Tối ưu Hyperparameters | Thử vài giá trị → chọn tốt nhất |
| 4 | 🤝 Ensemble & Stacking | Kết hợp models để cải thiện kết quả |

### 🎯 Mục tiêu

- Hiểu được **4 loại model** từ đơn giản → phức tạp
- Biết cách **tune hyperparameters** cơ bản
- Biết cách **kết hợp nhiều models** (ensemble/stacking)
- Đạt CV RMSE **< 0.13** với ít nhất 1 model

> 💡 **Lưu ý:** Tất cả code đều **đơn giản và step-by-step** — chạy từng cell theo thứ tự!

In [1]:
# ============================================================
# ♻️ SETUP — Import thư viện & rebuild pipeline từ Buổi 5
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# --- Linear models ---
from sklearn.linear_model import RidgeCV, LassoCV, Ridge

# --- Tree-based models ---
from sklearn.ensemble import RandomForestRegressor, StackingRegressor

# --- XGBoost (cần cài: pip install xgboost) ---
try:
    from xgboost import XGBRegressor
    HAVE_XGB = True
    print("✅ XGBoost sẵn sàng")
except ImportError:
    HAVE_XGB = False
    print("⚠️  XGBoost chưa cài — cài bằng: pip install xgboost")

# --- Đánh giá ---
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import mean_squared_error, r2_score

pd.set_option('display.float_format', lambda x: f'{x:.4f}')

# ── Load dữ liệu ──────────────────────────────────────────
train_df = pd.read_csv('data/train.csv')
test_df  = pd.read_csv('data/test.csv')

train_id = train_df['Id'].copy()
test_id  = test_df['Id'].copy()
y = np.log1p(train_df['SalePrice'])

train_df.drop(['Id', 'SalePrice'], axis=1, inplace=True)
test_df.drop(['Id'], axis=1, inplace=True)

ntrain = len(train_df)
ntest  = len(test_df)
all_data = pd.concat([train_df, test_df], axis=0, ignore_index=True)

# ── Fill missing ──────────────────────────────────────────
none_cols = ['PoolQC','MiscFeature','Alley','Fence','FireplaceQu',
             'GarageType','GarageFinish','GarageQual','GarageCond',
             'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','MasVnrType']
zero_cols = ['GarageYrBlt','GarageArea','GarageCars','MasVnrArea',
             'BsmtFinSF1','BsmtFinSF2','BsmtUnfSF','TotalBsmtSF','BsmtFullBath','BsmtHalfBath']
mode_cols = ['MSZoning','Utilities','Functional','Electrical',
             'KitchenQual','Exterior1st','Exterior2nd','SaleType']

for col in none_cols: all_data[col] = all_data[col].fillna('None')
for col in zero_cols: all_data[col] = all_data[col].fillna(0)
all_data['LotFrontage'] = (all_data.groupby('Neighborhood')['LotFrontage']
                           .transform(lambda x: x.fillna(x.median())))
for col in mode_cols: all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

# ── Xoá outlier ──────────────────────────────────────────
outlier_mask = (all_data['GrLivArea'][:ntrain] > 4000) & (np.expm1(y) < 300_000)
outlier_idx  = outlier_mask[outlier_mask].index.tolist()
all_data.drop(index=outlier_idx, inplace=True)
all_data.reset_index(drop=True, inplace=True)
y.drop(index=outlier_idx, inplace=True)
y.reset_index(drop=True, inplace=True)
ntrain = ntrain - len(outlier_idx)

# ── Feature Engineering ───────────────────────────────────
all_data['TotalSF']      = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']
all_data['TotalPorchSF'] = (all_data['OpenPorchSF'] + all_data['3SsnPorch'] +
                             all_data['EnclosedPorch'] + all_data['ScreenPorch'] + all_data['WoodDeckSF'])
all_data['TotalBath']    = (all_data['FullBath'] + 0.5*all_data['HalfBath'] +
                             all_data['BsmtFullBath'] + 0.5*all_data['BsmtHalfBath'])
all_data['OverallScore'] = all_data['OverallQual'] * all_data['OverallCond']
all_data['HasGarage']    = (all_data['GarageArea'] > 0).astype(int)
all_data['HasBsmt']      = (all_data['TotalBsmtSF'] > 0).astype(int)
all_data['HouseAge']     = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodAge']     = all_data['YrSold'] - all_data['YearRemodAdd']

for feat in ['OverallQual', 'GrLivArea', 'TotalSF']:
    all_data[f'{feat}_sq']   = all_data[feat] ** 2
    all_data[f'{feat}_sqrt'] = np.sqrt(all_data[feat])

for feat in ['GrLivArea', 'TotalSF', 'LotArea']:
    all_data[f'{feat}_log'] = np.log1p(all_data[feat])

# ── Ordinal Encoding ──────────────────────────────────────
ordinal_mappings = {
    'ExterQual':    ['None','Po','Fa','TA','Gd','Ex'],
    'ExterCond':    ['None','Po','Fa','TA','Gd','Ex'],
    'BsmtQual':     ['None','Po','Fa','TA','Gd','Ex'],
    'BsmtCond':     ['None','Po','Fa','TA','Gd','Ex'],
    'BsmtExposure': ['None','No','Mn','Av','Gd'],
    'BsmtFinType1': ['None','Unf','LwQ','Rec','BLQ','ALQ','GLQ'],
    'BsmtFinType2': ['None','Unf','LwQ','Rec','BLQ','ALQ','GLQ'],
    'HeatingQC':    ['None','Po','Fa','TA','Gd','Ex'],
    'KitchenQual':  ['None','Po','Fa','TA','Gd','Ex'],
    'FireplaceQu':  ['None','Po','Fa','TA','Gd','Ex'],
    'GarageFinish': ['None','Unf','RFn','Fin'],
    'GarageQual':   ['None','Po','Fa','TA','Gd','Ex'],
    'GarageCond':   ['None','Po','Fa','TA','Gd','Ex'],
    'PoolQC':       ['None','Fa','TA','Gd','Ex'],
    'Fence':        ['None','MnWw','GdWo','MnPrv','GdPrv'],
    'Functional':   ['Sal','Sev','Maj2','Maj1','Mod','Min2','Min1','Typ'],
    'LandSlope':    ['Sev','Mod','Gtl'],
    'LotShape':     ['IR3','IR2','IR1','Reg'],
    'PavedDrive':   ['N','P','Y'],
}
for col, order in ordinal_mappings.items():
    mapping = {val: i for i, val in enumerate(order)}
    all_data[col] = all_data[col].map(mapping).fillna(0).astype(int)

# ── One-Hot + Variance Filter + Scale ────────────────────
cat_cols     = all_data.select_dtypes(include=['object']).columns.tolist()
all_data_enc = pd.get_dummies(all_data, columns=cat_cols, drop_first=True)

X_train_raw = all_data_enc[:ntrain].copy()
X_test_raw  = all_data_enc[ntrain:].copy()
X_test_raw.reset_index(drop=True, inplace=True)

selector     = VarianceThreshold(threshold=0.01)
selected_mask = selector.fit(X_train_raw).get_support()
X_train_sel  = X_train_raw.loc[:, selected_mask]
X_test_sel   = X_test_raw.loc[:, selected_mask]

scaler  = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train_sel),
                        columns=X_train_sel.columns)
X_test  = pd.DataFrame(scaler.transform(X_test_sel),
                        columns=X_test_sel.columns)

# ── Helper: tính CV RMSE nhanh ───────────────────────────
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def rmse_cv(model):
    """Trả về CV RMSE trung bình (5-fold)."""
    scores = cross_val_score(model, X_train, y,
                             scoring='neg_root_mean_squared_error', cv=kf)
    return -scores.mean()

print("✅ Pipeline sẵn sàng!")
print(f"   X_train : {X_train.shape}")
print(f"   X_test  : {X_test.shape}")
print(f"   y       : {y.shape}   (log SalePrice)")

✅ XGBoost sẵn sàng
✅ Pipeline sẵn sàng!
   X_train : (1458, 160)
   X_test  : (1459, 160)
   y       : (1458,)   (log SalePrice)


**Expected Output** 

![img6-1](images/img_buoi6/img6-1.png)

---

## 📐 Task 1 — Linear Models: Bắt đầu với Baseline

Linear models là điểm khởi đầu tốt vì:
- **Nhanh** — train xong trong vài giây
- **Dễ hiểu** — mỗi feature có 1 hệ số rõ ràng
- **Là baseline** — giúp đánh giá tree models sau này có tốt hơn không

### Hai models sẽ dùng

| Model | Regularization | Đặc điểm |
|-------|---------------|---------|
| **Ridge** | L2 (bình phương hệ số) | Giữ tất cả features, hệ số nhỏ dần |
| **Lasso** | L1 (giá trị tuyệt đối) | Tự động **loại bỏ** features không cần thiết |

> **Tại sao cần Regularization?**
> Không có regularization → model dễ bị overfit (học vẹt training data).
> Regularization thêm "hình phạt" cho hệ số lớn → buộc model đơn giản hơn.


---

### 🔬 Hiểu Sâu Hơn: Cơ Chế Học của Ridge & Lasso

#### 📌 Hồi quy tuyến tính thông thường (không regularization)

Model tuyến tính học bằng cách **tối thiểu hóa hàm mất mát (Loss Function)**:

$$\text{Loss} = \sum_{i=1}^{n}(y_i - \hat{y}_i)^2 = \sum_{i=1}^{n}(y_i - \mathbf{w}^T\mathbf{x}_i)^2$$

Trong đó:
- $y_i$ = giá trị thực tế
- $\hat{y}_i = \mathbf{w}^T\mathbf{x}_i$ = giá trị model dự đoán
- $\mathbf{w}$ = vector hệ số (weights) mà model cần học

**Vấn đề:** Với nhiều features (như dataset nhà đất — ~200 features), model dễ **overfit** — hệ số $w$ trở nên rất lớn, model "học vẹt" training data thay vì học pattern thực sự.

---

#### 🔵 Ridge (L2 Regularization) — "Phạt hệ số lớn"

Ridge thêm vào hàm mất mát một **hình phạt L2**:

$$\text{Loss}_{\text{Ridge}} = \underbrace{\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}_{\text{sai số dự đoán}} + \underbrace{\alpha \sum_{j=1}^{p} w_j^2}_{\text{hình phạt L2}}$$

**Cơ chế hoạt động:**
- Khi $w_j$ lớn → hình phạt $w_j^2$ rất lớn → model bị "phạt nặng"
- Model phải **cân bằng**: vừa dự đoán đúng, vừa giữ hệ số nhỏ
- Kết quả: **tất cả** hệ số đều bị thu nhỏ (shrink về 0), nhưng **không ai = 0 hoàn toàn**
- $\alpha$ (alpha) kiểm soát mức độ phạt: $\alpha$ lớn → hệ số nhỏ hơn → model đơn giản hơn

---

#### 🟠 Lasso (L1 Regularization) — "Tự loại bỏ features"

Lasso thay hình phạt bằng **L1** (giá trị tuyệt đối):

$$\text{Loss}_{\text{Lasso}} = \underbrace{\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}_{\text{sai số dự đoán}} + \underbrace{\alpha \sum_{j=1}^{p} |w_j|}_{\text{hình phạt L1}}$$

**Tại sao L1 lại tạo ra hệ số = 0?**

Đây là điểm khác biệt quan trọng nhất:
- **L2** ($w^2$): Đạo hàm tại $w=0$ là $2w = 0$ → không có "lực đẩy" đặc biệt về 0
- **L1** ($|w|$): Đạo hàm tại $w>0$ là $+\alpha$, tại $w<0$ là $-\alpha$ → luôn có "lực kéo" cố định về 0

→ Khi hệ số $w_j$ nhỏ và không giúp ích nhiều, **Lasso sẽ kéo thẳng về 0** (loại bỏ feature đó)!

---

#### ⚙️ Tham số `alpha` ảnh hưởng thế nào?

| `alpha` | Hệ số $w$ | Model | Nguy cơ |
|---------|-----------|-------|---------|
| = 0 | Tự do, có thể rất lớn | Phức tạp | Overfit cao |
| Nhỏ (0.001) | Hơi bị thu nhỏ | Gần như linear thường | Vẫn có thể overfit |
| Vừa (10–100) | Thu nhỏ đáng kể | Cân bằng tốt | Ổn |
| Rất lớn | Gần = 0 hết | Quá đơn giản | Underfit |

> **`RidgeCV` và `LassoCV`** tự động thử nhiều giá trị `alpha` qua cross-validation → chọn `alpha` tối ưu nhất cho dữ liệu của bạn!


In [ ]:

# ── Visualization: Cơ chế của Ridge vs Lasso ─────────────
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ─── Panel 1: Hình phạt L1 vs L2 theo giá trị w ──────────
ax = axes[0]
w_range = np.linspace(-2, 2, 300)
l1 = np.abs(w_range)
l2 = w_range ** 2
ax.plot(w_range, l2, color='#2980b9', lw=2.5, label='L2 = $w^2$ (Ridge)')
ax.plot(w_range, l1, color='#e67e22', lw=2.5, label='L1 = $|w|$ (Lasso)')
ax.axvline(0, color='gray', linestyle='--', lw=1, alpha=0.5)
ax.set_xlabel('Giá trị hệ số w', fontsize=11)
ax.set_ylabel('Hình phạt (Penalty)', fontsize=11)
ax.set_title('So sánh hình phạt\nL1 (Lasso) vs L2 (Ridge)', fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(-0.1, 2.5)
ax.set_xlim(-2, 2)
ax.grid(True, alpha=0.3)
ax.annotate('L2 mềm mại\ngần w=0', xy=(0.3, 0.09), fontsize=9,
            color='#2980b9', fontweight='bold')
ax.annotate('L1 có "góc nhọn"\n→ kéo w về 0', xy=(0.35, 0.9), fontsize=9,
            color='#e67e22', fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ─── Panel 2: Ảnh hưởng của alpha lên hệ số (Ridge) ──────
ax = axes[1]
# Giả lập: hệ số Ridge thu nhỏ dần khi alpha tăng
alphas = np.logspace(-2, 3, 200)
# w_ridge ∝ 1/(1 + alpha) (simplified)
w_feature1 = 1.0 / (1 + alphas / 10)   # feature quan trọng
w_feature2 = 0.3 / (1 + alphas / 10)   # feature ít quan trọng
w_feature3 = 0.05 / (1 + alphas / 10)  # feature rất ít quan trọng
ax.semilogx(alphas, w_feature1, color='#27ae60', lw=2.5, label='Feature quan trọng')
ax.semilogx(alphas, w_feature2, color='#f39c12', lw=2.5, label='Feature vừa')
ax.semilogx(alphas, w_feature3, color='#e74c3c', lw=2.5, label='Feature ít quan trọng')
ax.axhline(0, color='black', lw=0.8, linestyle='-', alpha=0.4)
ax.axvline(50, color='gray', linestyle=':', lw=1.5, alpha=0.7, label='Alpha tốt (ví dụ)')
ax.set_xlabel('Alpha (log scale)', fontsize=11)
ax.set_ylabel('Hệ số w', fontsize=11)
ax.set_title('Ridge: Alpha càng lớn\nhệ số thu nhỏ dần (không về 0)', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ─── Panel 3: Ảnh hưởng của alpha lên hệ số (Lasso) ──────
ax = axes[2]
# Lasso: hệ số bị đưa về đúng 0 khi alpha đủ lớn (soft-thresholding)
def lasso_coef(w_true, alpha_arr, scale=10):
    """Giả lập soft-thresholding của Lasso"""
    threshold = alpha_arr / scale
    return np.maximum(0, np.abs(w_true) - threshold) * np.sign(w_true)

ax.semilogx(alphas, lasso_coef(1.0, alphas),   color='#27ae60', lw=2.5, label='Feature quan trọng')
ax.semilogx(alphas, lasso_coef(0.3, alphas),   color='#f39c12', lw=2.5, label='Feature vừa')
ax.semilogx(alphas, lasso_coef(0.05, alphas),  color='#e74c3c', lw=2.5, label='Feature ít quan trọng')
ax.axhline(0, color='black', lw=0.8, linestyle='-', alpha=0.4)
ax.axvline(50, color='gray', linestyle=':', lw=1.5, alpha=0.7, label='Alpha tốt (ví dụ)')

# Đánh dấu vùng "hệ số = 0"
ax.fill_between(alphas, -0.02, 0.02, alpha=0.15, color='red', label='Vùng hệ số ≈ 0')

ax.set_xlabel('Alpha (log scale)', fontsize=11)
ax.set_ylabel('Hệ số w', fontsize=11)
ax.set_title('Lasso: Alpha đủ lớn → hệ số = 0\n(tự loại bỏ feature!)', fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.suptitle('Cơ Chế Ridge vs Lasso — Regularization hoạt động thế nào?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("📌 Kết luận:")
print("  Ridge: TẤT CẢ hệ số thu nhỏ dần → không ai về đúng 0")
print("  Lasso: Hệ số nhỏ bị đẩy thẳng về 0 → tự động loại bỏ features không cần thiết")
print("  → Đây là lý do Lasso còn gọi là phương pháp 'Feature Selection tự động'!")


**Expected Output:**

![img6-2](images/img_buoi6/img6-2.png)
![img6-3](images/img_buoi6/img6-3.png)

In [ ]:
# ── Task 1: Train Ridge và Lasso ──────────────────────────
# RidgeCV tự tìm alpha tốt nhất qua cross-validation
ridge = RidgeCV(alphas=[1, 10, 50, 100, 300, 500])
#alpha của ridge thường lớn hơn lasso vì nó chỉ thu nhỏ hệ số chứ không đẩy về 0, nên phạm vi alpha được chọn cũng lớn hơn.
ridge.fit(X_train, y)
ridge_rmse = rmse_cv(ridge)
print(f"Ridge  → CV RMSE = {ridge_rmse:.4f}   (alpha tốt nhất = {ridge.alpha_:.0f})")

# LassoCV tương tự
lasso = LassoCV(alphas=[0.0001, 0.0003, 0.001, 0.003, 0.01],
                cv=kf, max_iter=5000)
#alpha của lasso thường nhỏ hơn nhiều so với ridge vì nó có thể đẩy hệ số về 0, nên phạm vi alpha được chọn cũng nhỏ hơn.
lasso.fit(X_train, y)
lasso_rmse = rmse_cv(lasso) 
#rmse là root mean squared error, đo lường sai số dự đoán của model trên tập huấn luyện qua cross-validation.
# Đơn vị đo lường thực tế: Vì đã được lấy căn bậc hai, RMSE có cùng đơn vị với biến mục tiêu. 
# Ví dụ, nếu bạn dự báo giá nhà theo đơn vị triệu đồng, thì RMSE là triệu đồng tức là sai số trung bình cũng nằm trên thang đo triệu đồng đó.
# Mức độ tin cậy: Nó cho biết trung bình mô hình của bạn dự đoán "sai" bao nhiêu so với thực tế. Càng nhỏ càng tốt.
nonzero = (lasso.coef_ != 0).sum()
print(f"Lasso  → CV RMSE = {lasso_rmse:.4f}   ({nonzero} features còn lại / {X_train.shape[1]} tổng)")

# ── Actual vs Predicted: xem model dự đoán "khớp" thực tế đến đâu ─
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, model, name, col in zip(
    axes, [ridge, lasso], ['Ridge', 'Lasso'], ['#2980b9', '#e67e22']
):
    y_pred = model.predict(X_train)
    ax.scatter(y, y_pred, alpha=0.25, s=14, color=col, label='Dự đoán')
    lim = [float(y.min()) - 0.1, float(y.max()) + 0.1]
    ax.plot(lim, lim, 'r--', lw=1.5, label='Lý tưởng (y = ŷ)')
    cv_val = ridge_rmse if name == 'Ridge' else lasso_rmse
    ax.text(0.05, 0.90, f'CV RMSE = {cv_val:.4f}', transform=ax.transAxes,
            fontsize=10, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.9))
    ax.set_xlabel('Giá thực tế — log(SalePrice)', fontsize=10)
    ax.set_ylabel('Giá dự đoán — log(SalePrice)', fontsize=10)
    ax.set_title(f'{name} — Actual vs Predicted', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.grid(True, alpha=0.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Linear Models — điểm càng gần đường đỏ = dự đoán càng chính xác',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n💡 Linear baseline tốt nhất: RMSE ≈ {min(ridge_rmse, lasso_rmse):.4f}")


**Expected Output:**

![img6-4](images/img_buoi6/img6-4.png)
![img6-5](images/img_buoi6/img6-5.png)

In [ ]:
# ── Lasso Feature Importance: Top 10 features quan trọng nhất ──
coef_df = pd.DataFrame({
    'Feature':     X_train.columns,
    'Coefficient': lasso.coef_
})
# Lasso tự loại features không cần thiết (hệ số = 0)
coef_df = coef_df[coef_df['Coefficient'] != 0].copy()
coef_df['Abs'] = coef_df['Coefficient'].abs()
top10 = coef_df.nlargest(10, 'Abs').sort_values('Coefficient')

# Màu: xanh = ảnh hưởng dương (tăng giá), đỏ = ảnh hưởng âm (giảm giá)
bar_colors = ['#e74c3c' if c < 0 else '#3498db' for c in top10['Coefficient']]

plt.figure(figsize=(9, 5))
bars = plt.barh(top10['Feature'], top10['Coefficient'],
                color=bar_colors, edgecolor='none')
plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
for bar, val in zip(bars, top10['Coefficient']):
    x_pos = val + 0.0015 if val >= 0 else val - 0.0015
    ha    = 'left'       if val >= 0 else 'right'
    plt.text(x_pos, bar.get_y() + bar.get_height() / 2,
             f'{val:+.4f}', va='center', ha=ha, fontsize=8)
plt.xlabel('Hệ số Lasso (Coefficient)')
plt.title('Top 10 Features quan trọng nhất (theo Lasso)', fontsize=12)
plt.tight_layout()
plt.show()

print(f"Lasso giữ lại {(lasso.coef_ != 0).sum()} / {X_train.shape[1]} features")
print("Các features có hệ số = 0 đã bị Lasso tự loại bỏ!")


**Expected Output:**

![img6-6](images/img_buoi6/img6-6.png)
![img6-7](images/img_buoi6/img6-7.png)

### 💡 Giải thích kết quả Linear Models

**Biểu đồ so sánh RMSE:**
- RMSE thấp hơn = model tốt hơn
- Ridge và Lasso thường cho kết quả gần nhau, RMSE khoảng **0.11 – 0.12** là bình thường

**Biểu đồ Feature Importance (Lasso):**
| Màu | Ý nghĩa |
|-----|---------|
| 🔵 Xanh | Hệ số dương → feature tăng → SalePrice **tăng** |
| 🔴 Đỏ | Hệ số âm → feature tăng → SalePrice **giảm** |

Ví dụ hợp lý: `TotalSF_log` dương (nhà to hơn → đắt hơn ✅), `HouseAge` âm (nhà cũ hơn → rẻ hơn ✅)

> **Lasso** tự động đưa hệ số về 0 với features không quan trọng — đây chính là tính năng **feature selection tự động** rất hữu ích trong thực tế!


---

✅ **Linear baseline xong!** Ridge và Lasso cho ta thấy features nào quan trọng nhất.

Tiếp theo ta thử **Tree-Based Models** — thường cho kết quả tốt hơn vì chúng nắm bắt được quan hệ **phi tuyến** (non-linear) mà linear models bỏ sót.


---

## 🌳 Task 2 — Tree-Based Models

### Random Forest là gì?

Thay vì train **1 cây** → train **100+ cây**, mỗi cây trên 1 phần dữ liệu ngẫu nhiên, rồi lấy **trung bình** kết quả.

```
Train set → Cây 1 → dự đoán 1
          → Cây 2 → dự đoán 2    →  Trung bình  →  Kết quả cuối
          → ...
          → Cây 100 → dự đoán 100
```

**Ưu điểm:** Ổn định hơn, ít overfit hơn 1 cây đơn.

### XGBoost là gì?

XGBoost dùng kỹ thuật **Gradient Boosting** — mỗi cây mới **học từ lỗi** của cây trước:

```
Cây 1 → dự đoán → lỗi 1
Cây 2 học từ lỗi 1 → dự đoán → lỗi 2
Cây 3 học từ lỗi 2 → ...
```

**Ưu điểm:** Thường cho kết quả tốt nhất trên bài toán tabular data!

In [ ]:
# ── Task 2a: Random Forest ────────────────────────────────
rf = RandomForestRegressor(
    n_estimators=100,    # 100 cây
    random_state=42,
    n_jobs=-1            # dùng tất cả CPU cores
)
rf.fit(X_train, y)
rf_rmse = rmse_cv(rf)
print(f"Random Forest → CV RMSE = {rf_rmse:.4f}")

**Expected Output:**

![img6-8](images/img_buoi6/img6-8.png)

### 💡 Giải thích kết quả Random Forest

- **`n_estimators=100`**: Train 100 cây quyết định, lấy trung bình kết quả → ổn định hơn 1 cây duy nhất
- **`n_jobs=-1`**: Dùng tất cả CPU cores → chạy nhanh hơn
- **`random_state=42`**: Đảm bảo kết quả tái tạo được (chạy lại vẫn ra số giống nhau)

> CV RMSE của RF thường cao hơn Linear Model với cấu hình mặc định — đừng lo, ta sẽ tune hyperparameters ở Task 3 để cải thiện!


In [ ]:
# ── Task 2b: XGBoost ─────────────────────────────────────
if not HAVE_XGB:
    print("⚠️  Cài XGBoost: pip install xgboost")
    xgb_rmse = None
else:
    xgb = XGBRegressor(
        n_estimators=300,      # 300 cây
        learning_rate=0.05,    # bước học nhỏ → kết quả ổn hơn
        max_depth=4,           # mỗi cây không quá sâu
        subsample=0.8,         # mỗi cây dùng 80% data
        random_state=42,
        verbosity=0
    )
    xgb.fit(X_train, y)
    xgb_rmse = rmse_cv(xgb)
    print(f"XGBoost       → CV RMSE = {xgb_rmse:.4f}")

# ── So sánh tất cả models đến giờ ────────────────────────
print("\n📊 Bảng so sánh:")
print(f"   Ridge         RMSE = {ridge_rmse:.4f}")
print(f"   Lasso         RMSE = {lasso_rmse:.4f}")
print(f"   Random Forest RMSE = {rf_rmse:.4f}")
if HAVE_XGB and xgb_rmse:
    print(f"   XGBoost       RMSE = {xgb_rmse:.4f}")

**Expected Output:**

![img6-9](images/img_buoi6/img6-9.png)

### 💡 So sánh 4 models đến giờ

| Model | RMSE thường thấy | Cách hoạt động |
|-------|-----------------|----------------|
| Ridge | ~0.115–0.120 | Hồi quy tuyến tính + phạt hệ số lớn (L2) |
| Lasso | ~0.115–0.120 | Hồi quy tuyến tính + tự loại bỏ features (L1) |
| Random Forest | ~0.130–0.145 | Trung bình 100 cây quyết định (chưa tune) |
| XGBoost | ~0.112–0.118 | Mỗi cây học từ lỗi cây trước (boosting) |

> **Random Forest nhiều khi tệ hơn Linear?** Có thể — vì 100 cây chưa đủ và chưa tune params.
> Task 3 tiếp theo sẽ **tối ưu hyperparameters** để cải thiện đáng kể!


---

## 🎛️ Task 3 — Tối Ưu Hyperparameters

**Hyperparameters** là các "núm điều chỉnh" của model — ta cần thử vài giá trị để tìm cái tốt nhất.

### Cách đơn giản nhất: thử từng giá trị trong một vòng lặp

```python
# Thử 4 giá trị → chọn cái có RMSE thấp nhất
for n in [100, 200, 300, 500]:
    model = RandomForestRegressor(n_estimators=n)
    rmse  = rmse_cv(model)
    print(f"n={n}: RMSE={rmse}")
```

Sau khi biết giá trị tốt nhất → **train lại 1 lần** với toàn bộ training data.

### Hyperparameters quan trọng cần tune

| Model | Param | Giải thích |
|-------|-------|-----------|
| Random Forest | `n_estimators` | Số cây — nhiều hơn thường tốt hơn, nhưng chậm hơn |
| XGBoost | `n_estimators` | Số cây |
| XGBoost | `learning_rate` | Bước học — nhỏ hơn thì chính xác hơn nhưng cần nhiều cây |
| XGBoost | `max_depth` | Độ sâu mỗi cây — sâu quá sẽ overfit |

In [ ]:
# ── Task 3a: Tune Random Forest ───────────────────────────
print("=== Tune Random Forest (n_estimators) ===")
best_rf_rmse = 999
best_n = 100
n_vals = [100, 150, 200, 300]
rf_rmses_list = []
for n in n_vals:
    m    = RandomForestRegressor(n_estimators=n, random_state=42, n_jobs=-1)
    rmse = rmse_cv(m)
    rf_rmses_list.append(rmse)
    mark = " ← tốt nhất" if rmse < best_rf_rmse else ""
    print(f"  n_estimators={n:>3}: RMSE={rmse:.4f}{mark}")
    if rmse < best_rf_rmse:
        best_rf_rmse = rmse
        best_n = n

rf_best = RandomForestRegressor(n_estimators=best_n, random_state=42, n_jobs=-1)
rf_best.fit(X_train, y)
print(f"\n✅ RF tốt nhất: n_estimators={best_n}, RMSE={best_rf_rmse:.4f}")

# ── Task 3b: Tune XGBoost learning_rate ──────────────────
lr_vals = [0.01, 0.05, 0.1]
xgb_rmses_list = []
if HAVE_XGB:
    print("\n=== Tune XGBoost (learning_rate) ===")
    best_xgb_rmse = 999
    best_lr = 0.05
    for lr in lr_vals:
        m    = XGBRegressor(n_estimators=300, learning_rate=lr,
                             max_depth=4, subsample=0.8,
                             random_state=42, verbosity=0)
        rmse = rmse_cv(m)
        xgb_rmses_list.append(rmse)
        mark = " ← tốt nhất" if rmse < best_xgb_rmse else ""
        print(f"  learning_rate={lr}: RMSE={rmse:.4f}{mark}")
        if rmse < best_xgb_rmse:
            best_xgb_rmse = rmse
            best_lr = lr

    xgb_best = XGBRegressor(n_estimators=300, learning_rate=best_lr,
                              max_depth=4, subsample=0.8,
                              random_state=42, verbosity=0)
    xgb_best.fit(X_train, y)
    print(f"\n✅ XGBoost tốt nhất: learning_rate={best_lr}, RMSE={best_xgb_rmse:.4f}")

# ── Biểu đồ line: RMSE thay đổi theo hyperparameter ──────
n_panels = 2 if HAVE_XGB else 1
fig, axes = plt.subplots(1, n_panels, figsize=(13 if HAVE_XGB else 6, 4))
if n_panels == 1:
    axes = [axes]

# Panel 1: RF — n_estimators vs RMSE
best_rf_idx = rf_rmses_list.index(min(rf_rmses_list))
axes[0].plot(n_vals, rf_rmses_list, 'o-', color='#27ae60', lw=2, markersize=9)
axes[0].scatter(n_vals[best_rf_idx], rf_rmses_list[best_rf_idx],
                color='#e74c3c', s=150, zorder=5, label=f'Tốt nhất (n={n_vals[best_rf_idx]})')
for x_v, y_v in zip(n_vals, rf_rmses_list):
    axes[0].annotate(f'{y_v:.4f}', (x_v, y_v),
                     textcoords='offset points', xytext=(0, 9), ha='center', fontsize=9)
axes[0].set_xlabel('n_estimators (số cây)', fontsize=10)
axes[0].set_ylabel('CV RMSE')
axes[0].set_title('Random Forest\nRMSE vs số cây', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(min(rf_rmses_list) * 0.985, max(rf_rmses_list) * 1.008)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

if HAVE_XGB and xgb_rmses_list:
    # Panel 2: XGBoost — learning_rate vs RMSE
    best_xgb_idx = xgb_rmses_list.index(min(xgb_rmses_list))
    axes[1].plot(range(len(lr_vals)), xgb_rmses_list, 's-', color='#e67e22', lw=2, markersize=9)
    axes[1].scatter(best_xgb_idx, xgb_rmses_list[best_xgb_idx],
                    color='#e74c3c', s=150, zorder=5,
                    label=f'Tốt nhất (lr={lr_vals[best_xgb_idx]})')
    for i, (lr_v, y_v) in enumerate(zip(lr_vals, xgb_rmses_list)):
        axes[1].annotate(f'{y_v:.4f}', (i, y_v),
                         textcoords='offset points', xytext=(0, 9), ha='center', fontsize=9)
    axes[1].set_xticks(range(len(lr_vals)))
    axes[1].set_xticklabels([str(lr) for lr in lr_vals])
    axes[1].set_xlabel('learning_rate', fontsize=10)
    axes[1].set_ylabel('CV RMSE')
    axes[1].set_title('XGBoost\nRMSE vs learning_rate', fontweight='bold')
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim(min(xgb_rmses_list) * 0.985, max(xgb_rmses_list) * 1.008)
    axes[1].spines['top'].set_visible(False)
    axes[1].spines['right'].set_visible(False)

plt.suptitle('Hyperparameter Tuning — điểm thấp nhất = cấu hình tốt nhất',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


**Expected Output:**

![img6-10](images/img_buoi6/img6-10.png)
![img6-11](images/img_buoi6/img6-11.png)

### 💡 Giải thích kết quả Hyperparameter Tuning

**Random Forest — `n_estimators`:**
- Tăng số cây → RMSE thường giảm, nhưng đến lúc nào đó sẽ không đổi nhiều (diminishing returns)
- Từ 200–300 cây trở lên thường đã đủ tốt

**XGBoost — `learning_rate`:**
- `learning_rate` lớn (0.1) → học nhanh nhưng có thể bỏ qua chi tiết → RMSE cao hơn
- `learning_rate` nhỏ (0.01) → học cẩn thận nhưng cần **nhiều cây hơn** để hội tụ
- Thực tế: dùng `learning_rate=0.05`, `n_estimators=300` là điểm cân bằng tốt

> **Nguyên tắc tuning:** Tune **từng param một**, giữ các param khác cố định. Tránh thay đổi nhiều thứ cùng lúc vì không biết cái gì gây ra sự thay đổi!


---

## 🤝 Task 4 — Ensemble & Stacking: Kết Hợp Models

**Ý tưởng:** Thay vì chọn 1 model tốt nhất, hãy **kết hợp nhiều models** lại!

### Cách 1 — Averaging (Trung bình đơn giản)

```
Model A: dự đoán [165k, 220k, 310k]
Model B: dự đoán [170k, 215k, 300k]
Trung bình: [167.5k, 217.5k, 305k]  ← thường tốt hơn cả hai!
```

### Cách 2 — Stacking (Xếp chồng)

```
Base models:  Ridge → y_pred_1
              RF    → y_pred_2
              XGB   → y_pred_3
                         ↓
Meta model:   Ridge học từ [y_pred_1, y_pred_2, y_pred_3]
                         ↓
                    Kết quả cuối
```

Stacking thông minh hơn averaging vì **meta model tự học** cách kết hợp tối ưu!

In [ ]:
# ── Task 4a: Averaging (trung bình đơn giản) ─────────────
ridge.fit(X_train, y)
rf_best.fit(X_train, y)

# Dự đoán trên training data
y_pred_ridge = ridge.predict(X_train)
y_pred_rf    = rf_best.predict(X_train)

# Trung bình 2 models (trọng số bằng nhau)
y_blend = (y_pred_ridge + y_pred_rf) / 2
blend_train_rmse = np.sqrt(mean_squared_error(y, y_blend))
print(f"Blend (Ridge + RF) — train RMSE = {blend_train_rmse:.4f}")

# RMSE thực sự qua CV (cần custom vì sklearn không hỗ trợ blend trực tiếp)
# Đơn giản: so sánh qua kết quả từng model riêng lẻ
print(f"\nGhi nhớ:")
print(f"  Ridge   CV RMSE = {ridge_rmse:.4f}")
print(f"  RF best CV RMSE = {best_rf_rmse:.4f}")
print("  Blend thường nằm giữa hoặc tốt hơn cả hai!")

**Expected Output:**

![img6-12](images/img_buoi6/img6-12.png)

### 💡 Giải thích Averaging (Blending)

Tại sao lấy **trung bình** lại tốt hơn chọn 1 model tốt nhất?

- Ridge giỏi ở một số nhà, RF giỏi ở một số nhà khác
- Trung bình cộng → lỗi của 2 model một phần **triệt tiêu lẫn nhau**
- Kết quả blend thường nằm **giữa hoặc tốt hơn cả hai**

> 📌 **Train RMSE thấp hơn CV RMSE** — điều này bình thường! Model đã "học" training data nên sai ít hơn. CV RMSE mới phản ánh khả năng dự đoán thực sự trên data mới.


In [ ]:

# ── Biểu đồ Averaging: so sánh Ridge / RF / Blend ────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

models_info = [
    (y_pred_ridge, 'Ridge',  '#2980b9'),
    (y_pred_rf,    'RF Best', '#27ae60'),
    (y_blend,      'Blend (Ridge + RF)', '#8e44ad'),
]

for ax, (y_pred, name, col) in zip(axes, models_info):
    rmse_val = np.sqrt(mean_squared_error(y, y_pred))
    ax.scatter(y, y_pred, alpha=0.25, s=14, color=col)
    lim = [float(y.min()) - 0.05, float(y.max()) + 0.05]
    ax.plot(lim, lim, 'r--', lw=1.5, label='Lý tưởng (y = ŷ)')
    ax.text(0.05, 0.90, f'Train RMSE = {rmse_val:.4f}',
            transform=ax.transAxes, fontsize=10, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.9))
    ax.set_xlabel('Giá thực tế — log(SalePrice)', fontsize=10)
    ax.set_ylabel('Giá dự đoán — log(SalePrice)', fontsize=10)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.grid(True, alpha=0.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Averaging — Blend (trung bình) thường cho điểm phân bố đồng đều hơn từng model riêng',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ── Bar chart: so sánh RMSE của 3 models ────────────────
blend_models = ['Ridge', 'RF Best', 'Blend\n(Ridge+RF)']
blend_rmses  = [
    np.sqrt(mean_squared_error(y, y_pred_ridge)),
    np.sqrt(mean_squared_error(y, y_pred_rf)),
    blend_train_rmse,
]
bar_cols = ['#2980b9', '#27ae60', '#8e44ad']

fig2, ax2 = plt.subplots(figsize=(7, 4))
bars = ax2.bar(blend_models, blend_rmses, color=bar_cols, edgecolor='white', width=0.5)
for bar, val in zip(bars, blend_rmses):
    ax2.text(bar.get_x() + bar.get_width() / 2, val + 0.001,
             f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax2.set_ylabel('Train RMSE', fontsize=11)
ax2.set_title('Blend vs Từng Model — RMSE càng thấp càng tốt', fontweight='bold')
ax2.set_ylim(0, max(blend_rmses) * 1.15)
ax2.grid(True, axis='y', alpha=0.3)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print("📌 Blend = trung bình đơn giản → lỗi của 2 model phần nào triệt tiêu nhau!")


**Expected Output:**

![img6-13](images/img_buoi6/img6-13.png)
![img6-14](images/img_buoi6/img6-14.png)

In [ ]:
# ── Task 4b: Stacking ─────────────────────────────────────
# Base models: 2-3 model mạnh nhất
if HAVE_XGB:
    base_models = [
        ('ridge', Ridge(alpha=50)),
        ('rf',    RandomForestRegressor(n_estimators=best_n, random_state=42, n_jobs=-1)),
        ('xgb',   XGBRegressor(n_estimators=300, learning_rate=best_lr,
                                max_depth=4, random_state=42, verbosity=0)),
    ]
else:
    base_models = [
        ('ridge', Ridge(alpha=50)),
        ('rf',    RandomForestRegressor(n_estimators=best_n, random_state=42, n_jobs=-1)),
    ]

# Meta model: Ridge đơn giản
meta_model = Ridge(alpha=10)

# StackingRegressor: sklearn tự lo việc cross-val bên trong
stacking = StackingRegressor(
    estimators      = base_models,
    final_estimator = meta_model,
    cv              = 5
)

stacking_rmse = rmse_cv(stacking)
stacking.fit(X_train, y)

print(f"Stacking → CV RMSE = {stacking_rmse:.4f}")
print(f"\n📊 Tổng kết Buổi 6:")
print(f"  Ridge         = {ridge_rmse:.4f}")
print(f"  Lasso         = {lasso_rmse:.4f}")
print(f"  Random Forest = {best_rf_rmse:.4f}")
if HAVE_XGB:
    print(f"  XGBoost       = {best_xgb_rmse:.4f}")
print(f"  Stacking      = {stacking_rmse:.4f}  ← thường tốt nhất!")

**Expected Output:**

![img6-15](images/img_buoi6/img6-15.png)

### 💡 Vì sao Stacking thường tốt hơn Averaging?

| Phương pháp | Cách kết hợp | Ưu điểm |
|------------|-------------|---------|
| Averaging | Trung bình đơn giản (trọng số = nhau) | Đơn giản, nhanh |
| Weighted Avg | Trung bình có trọng số (model tốt hơn → weight lớn hơn) | Linh hoạt hơn |
| **Stacking** | Meta model **tự học** cách kết hợp tối ưu | Thường tốt nhất |

**Stacking hoạt động thế nào trong `sklearn`?**
1. Chia training data thành K fold
2. Mỗi base model train trên K-1 fold, dự đoán fold còn lại
3. Dự đoán từng model trở thành **features mới** cho meta model
4. Meta model (Ridge) học cách kết hợp → kết quả cuối

> `StackingRegressor` của sklearn tự lo tất cả bước trên — ta chỉ cần truyền vào danh sách base models và meta model!


In [ ]:

# ── Biểu đồ Stacking: so sánh base models vs stacking ────
y_pred_stacking = stacking.predict(X_train)

# Panel 1: Scatter — mỗi model vs actual
n_base = 3 if HAVE_XGB else 2
models_stack = [
    (y_pred_ridge,    'Ridge (base)',   '#2980b9'),
    (y_pred_rf,       'RF Best (base)', '#27ae60'),
]
if HAVE_XGB:
    y_pred_xgb = xgb_best.predict(X_train)
    models_stack.append((y_pred_xgb, 'XGBoost (base)', '#e67e22'))
models_stack.append((y_pred_stacking, 'Stacking (meta)', '#8e44ad'))

fig, axes = plt.subplots(1, len(models_stack), figsize=(5 * len(models_stack), 5), sharey=True)

for ax, (y_pred, name, col) in zip(axes, models_stack):
    rmse_val = np.sqrt(mean_squared_error(y, y_pred))
    is_stack = 'Stacking' in name
    ax.scatter(y, y_pred, alpha=0.2, s=12, color=col)
    lim = [float(y.min()) - 0.05, float(y.max()) + 0.05]
    ax.plot(lim, lim, 'r--', lw=1.5)
    ax.text(0.05, 0.90, f'Train RMSE\n= {rmse_val:.4f}',
            transform=ax.transAxes, fontsize=10, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3',
                      facecolor='#d5f5e3' if is_stack else 'lightyellow', alpha=0.9))
    ax.set_xlabel('Giá thực tế — log(SalePrice)', fontsize=9)
    ax.set_ylabel('Giá dự đoán — log(SalePrice)', fontsize=9)
    title_sfx = '\n★ Meta Learner' if is_stack else ''
    ax.set_title(name + title_sfx, fontsize=10, fontweight='bold',
                 color='#6c3483' if is_stack else 'black')
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.grid(True, alpha=0.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Stacking — Meta model học cách kết hợp kết quả của các base models',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ── Bar chart: RMSE tất cả models trong stacking pipeline ─
stack_names = ['Ridge\n(base)', 'RF Best\n(base)']
stack_rmses = [
    np.sqrt(mean_squared_error(y, y_pred_ridge)),
    np.sqrt(mean_squared_error(y, y_pred_rf)),
]
stack_cols = ['#2980b9', '#27ae60']

if HAVE_XGB:
    stack_names.append('XGBoost\n(base)')
    stack_rmses.append(np.sqrt(mean_squared_error(y, y_pred_xgb)))
    stack_cols.append('#e67e22')

stack_names.append('Stacking\n(final)')
stack_rmses.append(np.sqrt(mean_squared_error(y, y_pred_stacking)))
stack_cols.append('#8e44ad')

fig2, ax2 = plt.subplots(figsize=(8, 4))
bars = ax2.bar(stack_names, stack_rmses, color=stack_cols, edgecolor='white', width=0.5)
# Đánh dấu cột Stacking
bars[-1].set_edgecolor('#6c3483')
bars[-1].set_linewidth(2.5)
for bar, val in zip(bars, stack_rmses):
    ax2.text(bar.get_x() + bar.get_width() / 2, val + 0.0005,
             f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax2.set_ylabel('Train RMSE', fontsize=11)
ax2.set_title('Stacking vs Base Models — Stacking học cách kết hợp tốt nhất', fontweight='bold')
ax2.set_ylim(0, max(stack_rmses) * 1.18)
ax2.grid(True, axis='y', alpha=0.3)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print("📌 Stacking: meta model tự học trọng số tối ưu cho từng base model")
print(f"   Base models RMSE:  Ridge={stack_rmses[0]:.4f} | RF={stack_rmses[1]:.4f}" +
      (f" | XGB={stack_rmses[2]:.4f}" if HAVE_XGB else ""))
print(f"   Stacking RMSE:     {stack_rmses[-1]:.4f}  ← tổng hợp tốt hơn từng model riêng lẻ!")


**Expected Output:**

![img6-16](images/img_buoi6/img6-16.png)
![img6-17](images/img_buoi6/img6-17.png)
![img6-18](images/img_buoi6/img6-18.png)

---

## 📝 Kiểm Tra Nhanh — Ôn Lại Kiến Thức Buổi 6

*Làm nhanh trong 5 phút — mỗi câu điền hoặc chọn 1 đáp án đúng.*

---

**Câu 1 (Điền từ):** Lasso dùng regularization dạng ___, Ridge dùng dạng ___.

> *(Gợi ý: L1 hoặc L2?)*

---

**Câu 2 (Trắc nghiệm):** Random Forest khác Decision Tree đơn lẻ ở điểm gì?

- A) RF chỉ dùng 1 cây nhưng sâu hơn
- B) RF dùng nhiều cây, mỗi cây train trên phần data ngẫu nhiên, rồi lấy trung bình
- C) RF không cần hyperparameter tuning
- D) RF sử dụng gradient descent

---

**Câu 3 (Điền từ):** Trong Stacking, **meta model** học cách ___ kết quả dự đoán của các base models.

---

**Câu 4 (Đúng/Sai):** Cross-validation RMSE thường **thấp hơn** Train RMSE.

> *(Đúng / Sai?)*

---

> 🔍 Xem đáp án và lời giải ở ô markdown bên dưới!


In [ ]:
# ── Thực Hành Nhanh: hoàn thiện 3 dòng bên dưới ──────────
# (điền vào chỗ ... để code chạy đúng)

# 1. Tạo Random Forest với 200 cây
rf_check = RandomForestRegressor(n_estimators=..., random_state=42, n_jobs=-1)

# 2. Tính CV RMSE (dùng hàm rmse_cv đã có sẵn)
rmse_check = rmse_cv(...)

# 3. In kết quả
print(f"RF (200 cây) → CV RMSE = {rmse_check:.4f}")


## ✅ Đáp Án & Lời Giải

**Câu 1:** Lasso → **L1**,  Ridge → **L2**
- **L1** (Lasso): phạt tổng giá trị tuyệt đối `Σ|wᵢ|` → đẩy một số hệ số về đúng **0** → tự loại bỏ feature thừa
- **L2** (Ridge): phạt tổng bình phương `Σwᵢ²` → thu nhỏ tất cả hệ số đều đặn, không ai về 0

---

**Câu 2:** **B** — RF dùng nhiều cây, mỗi cây train trên phần data ngẫu nhiên, rồi lấy trung bình
- Kỹ thuật này gọi là **Bootstrap Aggregating (Bagging)** — giúp giảm variance, model ổn định hơn 1 cây đơn lẻ.

---

**Câu 3:** Meta model học cách **kết hợp tối ưu** (combine / weight)
- Khác với Averaging (trung bình đơn giản), meta model tìm trọng số tốt nhất cho từng base model tùy theo từng tình huống → thường cho kết quả tốt hơn.

---

**Câu 4:** **Sai** — CV RMSE thường **cao hơn** Train RMSE
- Model đã "nhìn thấy" training data trong quá trình học → sai ít hơn trên chính data đó.
- CV RMSE đánh giá trên data **model chưa từng thấy** → trung thực hơn, và thường cao hơn.

---

**Thực Hành:** Code hoàn chỉnh:
```python
rf_check = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rmse_check = rmse_cv(rf_check)
```

---

## 🏁 Tổng Kết Buổi 6

| Chủ đề | Nội dung cốt lõi |
|--------|-----------------|
| **Linear Models** | Ridge (L2) + Lasso (L1) — baseline nhanh, dễ giải thích |
| **Tree Models** | RF (bagging) + XGBoost (boosting) — thường RMSE thấp hơn |
| **Hyperparameter Tuning** | Thử từng giá trị trong vòng lặp, chọn RMSE thấp nhất |
| **Ensemble / Stacking** | Kết hợp nhiều models qua meta-learner — tốt nhất thực tế |

### ➡️ Buổi 7 — Đánh Giá Model Toàn Diện

Buổi tới ta sẽ đi sâu vào **evaluation**:
- Metrics: RMSE, MAE, R² — ý nghĩa và khi nào dùng cái nào?
- Learning curves — model đang bị bias (underfit) hay variance (overfit)?
- Residual analysis — model còn thiếu sót ở đâu?
- So sánh tất cả models side-by-side
